### PACOTES

In [16]:
import pandas as pd
import numpy as np
import time

from itertools import combinations

from scipy.stats import (
    ks_2samp,
    wasserstein_distance,
    spearmanr
)

from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    roc_auc_score,
    mutual_info_score
)

from sklearn.preprocessing import RobustScaler


from joblib import Parallel, delayed

from IPython.display import display 



In [12]:

inicio = time.time()
df_resumo = pd.read_csv('visu_resumo_features.csv')
df = pd.read_csv('creditcard.csv')

# AJUSTE DE TIPOS
df_resumo['Comparacao_dis_whith'] = pd.to_numeric(
    df_resumo['Comparacao_dis_whith'],
    errors='coerce'
)

# MANN-WHITNEY SCORE
# Quanto menor o p-value -> melhor
df_resumo['MannWhitney_Score'] = -np.log10(
    df_resumo['Comparacao_dis_whith'] + 1e-300
)

# CORRELAÇÃO ABSOLUTA
df_resumo['Corr_spearman_abs'] = abs(
    df_resumo['Correlacao_spearman']
)

# AJUSTE DE MÉTRICAS DISTRIBUCIONAIS

# KL e Wasser podem explodir
# Aplicamos log para estabilizar

df_resumo['divergencia_kl'] = np.log1p(
    abs(df_resumo['divergencia_kl'])
)

df_resumo['Wasser'] = np.log1p(
    abs(df_resumo['Wasser'])
)

# AJUSTE DE NORMALIDADE

# Quanto MENOR distância da normalidade -> melhor
df_resumo['Normalidade_Score'] = 1 / (
    1 + abs(df_resumo['Normalidade'])
)

# AJUSTE DE ASSIMETRIA
df_resumo['Skewness_Score'] = 1 / (
    1 + abs(df_resumo['Assimetria'])
)


# AJUSTE DE CURTOSE
df_resumo['Kurtosis_Score'] = 1 / (
    1 + abs(df_resumo['Curtose'])
)

# AJUSTE DE OUTLIERS
df_resumo['Outlier_Score'] = 1 / (
    1 + df_resumo['Qnd_Outliers']
)

# MÉTRICAS UTILIZADAS
metricas = [

    # INDISPENSÁVEL
    'Curva ROC',
    'KS',

    # MUITO IMPORTANTE
    'divergencia_kl',
    'Wasser',

    # IMPORTANTE
    'inf_mutua',
    'MannWhitney_Score',

    # BOM
    'Normalidade_Score',
    'Sep_mediana_norm',

    # ÚTIL
    'Skewness_Score',
    'Kurtosis_Score',

    # ACESSÓRIO
    'Corr_spearman_abs',
    'Outlier_Score'
]


# TRATAMENTO DE NaN
df_resumo[metricas] = df_resumo[
    metricas
].fillna(0)


# NORMALIZAÇÃO ROBUSTA
# Melhor para fraude/outliers

scaler = RobustScaler()

df_norm = df_resumo.copy()

df_norm[metricas] = scaler.fit_transform(
    df_norm[metricas]
)


# MIN-MAX FINAL
# Após robust scaling

for col in metricas:

    minimo = df_norm[col].min()
    maximo = df_norm[col].max()

    df_norm[col] = (
        (df_norm[col] - minimo)
        /
        (maximo - minimo + 1e-9)
    )


# SCORE FINAL
df_norm['Score_Final'] = (

    # INDISPENSÁVEL -> 0.25
    df_norm['Curva ROC'] * 0.125 +
    df_norm['KS'] * 0.125 +

    # MUITO IMPORTANTE -> 0.21
    df_norm['divergencia_kl'] * 0.105 +
    df_norm['Wasser'] * 0.105 +

    # IMPORTANTE -> 0.17
    df_norm['inf_mutua'] * 0.085 +
    df_norm['MannWhitney_Score'] * 0.085 +

    # BOM -> 0.13
    df_norm['Normalidade_Score'] * 0.065 +
    df_norm['Sep_mediana_norm'] * 0.065 +

    # ÚTIL -> 0.09
    df_norm['Skewness_Score'] * 0.045 +
    df_norm['Kurtosis_Score'] * 0.045 +

    # ACESSÓRIO -> 0.05
    df_norm['Corr_spearman_abs'] * 0.025 +
    df_norm['Outlier_Score'] * 0.025
)


# GARANTIR SEM NaN
df_norm['Score_Final'] = df_norm[
    'Score_Final'
].fillna(0)

# RANKING FINAL
ranking = df_norm.sort_values(
    'Score_Final',
    ascending=False
).reset_index(drop=True)

# POSIÇÃO
ranking.insert(
    0,
    'Posicao_Rank',
    ranking.index + 1
)

# EXIBIÇÃO
pd.set_option(
    'display.max_columns',
    None
)

display(ranking)

# SALVAR CSV
ranking.to_csv(
    '1x1visu_scores.csv',
    index=False
)

print('\nCSV salvo com sucesso!')

,Posicao_Rank,Feature,Normalidade,Qnd_Outliers,Sep_mediana_norm,Comparacao_dis_whith,Correlacao_spearman,inf_mutua,divergencia_kl,Curva ROC,KS,Wasser,Assimetria,Curtose,MannWhitney_Score,Corr_spearman_abs,Normalidade_Score,Skewness_Score,Kurtosis_Score,Outlier_Score,Score_Final
0,1,V14,0,14149,1.000000,1.471581e-260,-0.064613,0.984641,1.000000,1.000000,1.000000,0.211410,1.995165,23.879022,1.000000,1.000000,0.0,0.305756,0.046824,0.000045,0.651760
1,2,V12,0,15348,0.796989,8.416027e-247,-0.062870,0.917286,0.840947,0.972118,0.924927,0.201544,2.278389,20.241493,0.946936,0.972125,0.0,0.274060,0.055062,0.000040,0.595970
2,3,V10,0,9496,0.579452,9.611131e-222,-0.059564,0.908347,0.894367,0.919245,0.950893,0.192730,1.187134,31.987656,0.850284,0.919255,0.0,0.441298,0.035000,0.000080,0.579477
3,4,V11,0,780,0.497847,4.910592e-226,0.060143,0.820345,0.725013,0.928507,0.889460,0.157954,0.356504,1.633872,0.866838,0.928514,0.0,0.748947,0.453109,0.001255,0.573065
4,5,V4,0,11148,0.415041,3.625904e-248,0.063045,0.586806,0.805397,0.974920,0.902628,0.173101,0.676289,2.635388,0.952203,0.974924,0.0,0.594410,0.327929,0.000064,0.561109
5,6,V17,0,7420,0.870823,9.219384e-124,-0.044335,1.000000,0.905760,0.675706,0.875582,0.212480,3.844894,94.798034,0.472352,0.675708,0.0,0.165685,0.011212,0.000109,0.517935
6,7,V3,0,3363,0.486416,1.211048e-219,-0.059278,0.583784,0.718940,0.914680,0.822726,0.212220,2.240144,26.619062,0.842183,0.914681,0.0,0.278017,0.042052,0.000272,0.505049
7,8,V16,0,8184,0.580474,1.808172e-156,-0.049936,0.733854,0.745415,0.765280,0.800845,0.165757,1.100960,10.418927,0.598510,0.765281,0.0,0.461906,0.103528,0.000097,0.486999
8,9,V7,0,8948,0.346706,1.464234e-146,-0.048308,0.457887,0.709440,0.739240,0.767815,0.191285,2.553894,405.600275,0.560292,0.739245,0.0,0.248076,0.001663,0.000086,0.421761
9,10,V9,0,8283,0.272512,8.943723e-154,-0.049499,0.498804,0.482208,0.758284,0.660477,0.127076,0.554677,3.731224,0.588117,0.758292,0.0,0.645688,0.251678,0.000095,0.410763



CSV salvo com sucesso!


### TOP 10


In [13]:

# CARREGAR CSV DO RANKING
ranking = pd.read_csv('1x1visu_scores.csv')

ranking = ranking.sort_values(
    by='Score_Final',
    ascending=False
)

top10 = ranking.head(10)

# ATRIBUINDO ÀS VARIÁVEIS
Primeiro_lugar = top10.iloc[0]['Feature']
Segundo_lugar = top10.iloc[1]['Feature']
Terceiro_lugar = top10.iloc[2]['Feature']
Quarto_lugar = top10.iloc[3]['Feature']
Quinto_lugar = top10.iloc[4]['Feature']
Sexto_lugar = top10.iloc[5]['Feature']
Setimo_lugar = top10.iloc[6]['Feature']
Oitavo_lugar = top10.iloc[7]['Feature']
Nono_lugar = top10.iloc[8]['Feature']
Decimo_lugar = top10.iloc[9]['Feature']

# PRINT TOP 10
print('TOP 10 FEATURES:\n')

for i, row in top10.iterrows():

    print(
        f"{row['Posicao_Rank']}º -> "
        f"{row['Feature']}"
    )

# LISTA FINAL
features_top10 = top10['Feature'].tolist()
print('\nLista Top 10:\n')
print(features_top10)

TOP 10 FEATURES:

1º -> V14
2º -> V12
3º -> V10
4º -> V11
5º -> V4
6º -> V17
7º -> V3
8º -> V16
9º -> V7
10º -> V9

Lista Top 10:

['V14', 'V12', 'V10', 'V11', 'V4', 'V17', 'V3', 'V16', 'V7', 'V9']


 ### CRIANDO CSV DE COMBINACOES 2X2 

In [14]:


# =========================================================
# TEMPO
# =========================================================

inicio = time.time()

# =========================================================
# DATASET
# =========================================================

df = pd.read_csv('creditcard.csv')

# =========================================================
# FRAUDE / NORMAL
# =========================================================

fraude = df[df['status_fraude'] == 1]
normal = df[df['status_fraude'] == 0]

df_visu = df.copy()

# =========================================================
# SAMPLE 5%
# APENAS PARA MÉTRICAS PESADAS
# =========================================================

df_cluster = df_visu.sample(
    frac=0.05,
    random_state=42
)

print(f'\nAmostra cluster: {len(df_cluster)} linhas')

# =========================================================
# FEATURES PCA
# =========================================================

features = [

    col for col in df.columns
    if col.startswith('V')

]

# =========================================================
# COMBINAÇÕES 2x2
# =========================================================

duplas = list(combinations(features, 2))

print(f'\nTotal de combinações: {len(duplas)}')

# =========================================================
# RESULTADOS
# =========================================================

resultados = []

# =========================================================
# LOOP PRINCIPAL
# =========================================================

for i, (f1, f2) in enumerate(duplas):

    try:

        print(
            f'\n[{i+1}/{len(duplas)}] '
            f'Processando: {f1} | {f2}'
        )

        # =================================================
        # DADOS COMPLETOS
        # =================================================

        y_full = df_visu['status_fraude'].values

        # =================================================
        # DADOS SAMPLE
        # =================================================

        X = df_cluster[[f1, f2]].values

        # =================================================
        # GMM
        # =================================================

        gmm = GaussianMixture(

            n_components=2,
            covariance_type='diag',
            random_state=42,
            max_iter=100

        )

        gmm.fit(X)

        clusters = gmm.predict(X)

        probs = gmm.predict_proba(X)

        # =================================================
        # SILHOUETTE
        # =================================================

        silhouette = silhouette_score(
            X,
            clusters
        )

        # =================================================
        # DAVIES-BOULDIN
        # =================================================

        db = davies_bouldin_score(
            X,
            clusters
        )

        db_invertido = 1 / (
            1 + db
        )

        # =================================================
        # DISTÂNCIA ENTRE CENTROIDES
        # =================================================

        centroides = gmm.means_

        dist_centroides = np.linalg.norm(
            centroides[0] - centroides[1]
        )

        # =================================================
        # OVERLAP GMM
        # =================================================

        overlap = np.mean(
            np.min(probs, axis=1)
        )

        overlap_score = 1 - overlap

        # =================================================
        # GEO SCORE
        # =================================================

        geo_score = (

            silhouette +
            db_invertido +
            dist_centroides

        ) / 3

        # =================================================
        # ROC
        # =================================================

        roc1 = roc_auc_score(
            y_full,
            np.abs(df_visu[f1])
        )

        roc2 = roc_auc_score(
            y_full,
            np.abs(df_visu[f2])
        )

        roc = (roc1 + roc2) / 2

        # =================================================
        # KS
        # =================================================

        ks1 = ks_2samp(
            fraude[f1],
            normal[f1]
        ).statistic

        ks2 = ks_2samp(
            fraude[f2],
            normal[f2]
        ).statistic

        ks = (ks1 + ks2) / 2

        # =================================================
        # WASSERSTEIN
        # =================================================

        wasser1 = wasserstein_distance(
            fraude[f1],
            normal[f1]
        )

        wasser2 = wasserstein_distance(
            fraude[f2],
            normal[f2]
        )

        wasser = (wasser1 + wasser2) / 2

        # =================================================
        # KL DIVERGENCE
        # =================================================

        hist_f1_fraud, bins = np.histogram(
            fraude[f1],
            bins=30,
            density=True
        )

        hist_f1_normal, _ = np.histogram(
            normal[f1],
            bins=bins,
            density=True
        )

        hist_f2_fraud, bins2 = np.histogram(
            fraude[f2],
            bins=30,
            density=True
        )

        hist_f2_normal, _ = np.histogram(
            normal[f2],
            bins=bins2,
            density=True
        )

        eps = 1e-10

        kl1 = np.sum(

            hist_f1_fraud *
            np.log(
                (hist_f1_fraud + eps)
                /
                (hist_f1_normal + eps)
            )

        )

        kl2 = np.sum(

            hist_f2_fraud *
            np.log(
                (hist_f2_fraud + eps)
                /
                (hist_f2_normal + eps)
            )

        )

        kl = (kl1 + kl2) / 2

        # =================================================
        # AJUSTE LOG
        # =================================================

        kl = np.log1p(abs(kl))
        wasser = np.log1p(abs(wasser))

        # =================================================
        # INFORMAÇÃO MÚTUA
        # =================================================

        mi1 = mutual_info_score(
            pd.qcut(df_visu[f1], q=10, duplicates='drop'),
            y_full
        )

        mi2 = mutual_info_score(
            pd.qcut(df_visu[f2], q=10, duplicates='drop'),
            y_full
        )

        mi = (mi1 + mi2) / 2

        # =================================================
        # ASSIMETRIA
        # =================================================

        skewness = (

            abs(df_visu[f1].skew()) +
            abs(df_visu[f2].skew())

        ) / 2

        skewness_score = 1 / (
            1 + skewness
        )

        # =================================================
        # CURTOSE
        # =================================================

        curtose = (

            abs(df_visu[f1].kurtosis()) +
            abs(df_visu[f2].kurtosis())

        ) / 2

        kurtosis_score = 1 / (
            1 + curtose
        )

        # =================================================
        # OUTLIERS
        # =================================================

        def contar_outliers(col):

            q1 = col.quantile(0.25)
            q3 = col.quantile(0.75)

            iqr = q3 - q1

            inf = q1 - 1.5 * iqr
            sup = q3 + 1.5 * iqr

            return (
                (col < inf) |
                (col > sup)
            ).sum()

        out1 = contar_outliers(df_visu[f1])
        out2 = contar_outliers(df_visu[f2])

        outliers = (out1 + out2) / 2

        outlier_score = 1 / (
            1 + outliers
        )

        # =================================================
        # CORRELAÇÃO
        # MENOR CORRELAÇÃO = MELHOR
        # =================================================

        corr, _ = spearmanr(
            df_visu[f1],
            df_visu[f2]
        )

        corr_score = 1 - abs(corr)

        # =================================================
        # SALVAR
        # =================================================

        resultados.append({

            'Features': f'{f1} | {f2}',

            # INDISPENSÁVEL
            'Curva ROC': roc,
            'KS': ks,

            # MUITO IMPORTANTE
            'divergencia_kl': kl,
            'Wasser': wasser,

            # IMPORTANTE
            'inf_mutua': mi,
            'Overlap_GMM': overlap_score,

            # BOM
            'Geo_Score': geo_score,

            # ÚTIL
            'Skewness_Score': skewness_score,
            'Kurtosis_Score': kurtosis_score,

            # ACESSÓRIO
            'Corr_Score': corr_score,
            'Outlier_Score': outlier_score,

            # EXTRA
            'Silhouette': silhouette,
            'Davies_Bouldin': db,
            'Dist_Centroides': dist_centroides

        })

    except Exception as e:

        print(f'\nErro em {f1} | {f2}')
        print(e)

# =========================================================
# DATAFRAME
# =========================================================

df_scores = pd.DataFrame(resultados)

# =========================================================
# MÉTRICAS
# =========================================================

metricas = [

    'Curva ROC',
    'KS',

    'divergencia_kl',
    'Wasser',

    'inf_mutua',
    'Overlap_GMM',

    'Geo_Score',

    'Skewness_Score',
    'Kurtosis_Score',

    'Corr_Score',
    'Outlier_Score'

]

# =========================================================
# NaN
# =========================================================

df_scores[metricas] = df_scores[
    metricas
].fillna(0)

# =========================================================
# ROBUST SCALER
# =========================================================

scaler = RobustScaler()

df_norm = df_scores.copy()

df_norm[metricas] = scaler.fit_transform(
    df_norm[metricas]
)

# =========================================================
# MINMAX
# =========================================================

for col in metricas:

    minimo = df_norm[col].min()
    maximo = df_norm[col].max()

    df_norm[col] = (

        (df_norm[col] - minimo)

        /

        (maximo - minimo + 1e-9)

    )

# =========================================================
# SCORE FINAL
# =========================================================

df_norm['Score_Final'] = (

    # INDISPENSÁVEL -> 0.25
    df_norm['Curva ROC'] * 0.125 +
    df_norm['KS'] * 0.125 +

    # MUITO IMPORTANTE -> 0.21
    df_norm['divergencia_kl'] * 0.105 +
    df_norm['Wasser'] * 0.105 +

    # IMPORTANTE -> 0.17
    df_norm['inf_mutua'] * 0.085 +
    df_norm['Overlap_GMM'] * 0.085 +

    # BOM -> 0.13
    df_norm['Geo_Score'] * 0.13 +

    # ÚTIL -> 0.09
    df_norm['Skewness_Score'] * 0.045 +
    df_norm['Kurtosis_Score'] * 0.045 +

    # ACESSÓRIO -> 0.05
    df_norm['Corr_Score'] * 0.025 +
    df_norm['Outlier_Score'] * 0.025

)

# =========================================================
# GARANTIR SEM NaN
# =========================================================

df_norm['Score_Final'] = df_norm[
    'Score_Final'
].fillna(0)

# =========================================================
# RANKING FINAL
# =========================================================

ranking = df_norm.sort_values(
    'Score_Final',
    ascending=False
).reset_index(drop=True)

# =========================================================
# POSIÇÃO
# =========================================================

ranking.insert(
    0,
    'Posicao_Rank',
    ranking.index + 1
)

# =========================================================
# EXIBIR
# =========================================================

pd.set_option(
    'display.max_columns',
    None
)

display(
    ranking.head(30)
)

# =========================================================
# SALVAR CSV
# =========================================================

ranking.to_csv(
    '2x2visu_scores.csv',
    index=False
)

# =========================================================
# TEMPO
# =========================================================

fim = time.time()

print(
    f'\nTempo total: {(fim - inicio)/60:.2f} minutos'
)

print('\nCSV salvo com sucesso!')


fim = time.time()

print(f"Tempo total: {fim - inicio:.4f} segundos")



Amostra cluster: 14240 linhas

Total de combinações: 378

[1/378] Processando: V1 | V2

[2/378] Processando: V1 | V3

[3/378] Processando: V1 | V4

[4/378] Processando: V1 | V5

[5/378] Processando: V1 | V6

[6/378] Processando: V1 | V7

[7/378] Processando: V1 | V8

[8/378] Processando: V1 | V9

[9/378] Processando: V1 | V10

[10/378] Processando: V1 | V11

[11/378] Processando: V1 | V12

[12/378] Processando: V1 | V13

[13/378] Processando: V1 | V14

[14/378] Processando: V1 | V15

[15/378] Processando: V1 | V16

[16/378] Processando: V1 | V17

[17/378] Processando: V1 | V18

[18/378] Processando: V1 | V19

[19/378] Processando: V1 | V20

[20/378] Processando: V1 | V21

[21/378] Processando: V1 | V22

[22/378] Processando: V1 | V23

[23/378] Processando: V1 | V24

[24/378] Processando: V1 | V25

[25/378] Processando: V1 | V26

[26/378] Processando: V1 | V27

[27/378] Processando: V1 | V28

[28/378] Processando: V2 | V3

[29/378] Processando: V2 | V4

[30/378] Processando: V2 | V5

[

,Posicao_Rank,Features,Curva ROC,KS,divergencia_kl,Wasser,inf_mutua,Overlap_GMM,Geo_Score,Skewness_Score,Kurtosis_Score,Corr_Score,Outlier_Score,Silhouette,Davies_Bouldin,Dist_Centroides,Score_Final
0,1,V10 | V14,0.983649,1.000000,0.822258,0.953081,0.971016,0.909102,0.064153,0.354995,0.041129,0.993325,0.060389,0.640331,2.639647,1.097518,0.646685
1,2,V12 | V14,0.976991,0.986642,0.904744,0.972621,0.976802,0.814629,0.072102,0.273888,0.052064,0.922681,0.042232,0.538373,1.741754,1.325176,0.643013
2,3,V10 | V11,0.926387,0.943132,0.908660,0.830806,0.874002,0.944358,0.088480,0.570614,0.067941,0.949676,0.179443,0.661639,1.600521,1.626458,0.639360
3,4,V14 | V17,1.000000,0.961256,0.830968,0.998080,0.929056,0.824791,0.063622,0.196928,0.018782,0.756160,0.069198,0.558572,1.897746,1.094538,0.624896
4,5,V11 | V17,0.942738,0.904388,0.915856,0.887563,0.832042,0.910253,0.072417,0.278370,0.023435,0.891488,0.232743,0.558174,1.511654,1.280504,0.619446
5,6,V10 | V16,0.930761,0.897544,0.896291,0.846660,0.823136,0.901779,0.074900,0.452230,0.054143,0.918978,0.091266,0.644487,2.049634,1.331785,0.615946
6,7,V10 | V17,0.960353,0.935992,0.787407,0.955856,0.854867,0.940926,0.060184,0.232344,0.017485,0.782194,0.096794,0.676095,3.555815,1.009316,0.613769
7,8,V11 | V14,0.966034,0.968396,0.942280,0.884382,0.948191,0.554942,0.029091,0.443997,0.088497,0.776638,0.113819,0.389545,4.601093,0.493241,0.611375
8,9,V3 | V10,0.899294,0.908801,0.658858,0.955181,0.847667,0.955988,0.134293,0.333948,0.039188,0.895484,0.137150,0.666492,1.516904,2.851793,0.608862
9,10,V10 | V12,0.937345,0.961379,0.868066,0.928094,0.902613,0.674758,0.064012,0.330833,0.044009,0.757426,0.055972,0.457035,1.630112,1.171533,0.605538



Tempo total: 48.40 minutos

CSV salvo com sucesso!
Tempo total: 2904.2608 segundos


 ### CRIANDO CSV DE COMBINACOES 3x3

In [20]:
# =========================================================
# FUNÇÃO WORKER (O QUE CADA NÚCLEO VAI EXECUTAR)
# =========================================================
def processar_tripla(f1, f2, f3, df_visu, df_cluster, fraude, normal):
    try:
        y_full = df_visu['status_fraude'].values
        X = df_cluster[[f1, f2, f3]].values

        # GMM
        gmm = GaussianMixture(n_components=2, covariance_type='diag', random_state=42, max_iter=100)
        gmm.fit(X)
        clusters = gmm.predict(X)
        probs = gmm.predict_proba(X)

        # SILHOUETTE E DAVIES-BOULDIN
        silhouette = silhouette_score(X, clusters)
        db = davies_bouldin_score(X, clusters)
        db_invertido = 1 / (1 + db)

        # DISTÂNCIA ENTRE CENTROIDES E OVERLAP GMM
        centroides = gmm.means_
        dist_centroides = np.linalg.norm(centroides[0] - centroides[1])
        overlap_score = 1 - np.mean(np.min(probs, axis=1))

        # GEO SCORE
        geo_score = (silhouette + db_invertido + dist_centroides) / 3

        # ROC, KS E WASSERSTEIN
        roc = np.mean([roc_auc_score(y_full, np.abs(df_visu[f])) for f in [f1, f2, f3]])
        ks = np.mean([ks_2samp(fraude[f], normal[f]).statistic for f in [f1, f2, f3]])
        wasser = np.mean([wasserstein_distance(fraude[f], normal[f]) for f in [f1, f2, f3]])

        # KL DIVERGENCE
        eps = 1e-10
        kl_values = []
        for feat in [f1, f2, f3]:
            hist_fraud, bins = np.histogram(fraude[feat], bins=30, density=True)
            hist_normal, _ = np.histogram(normal[feat], bins=bins, density=True)
            kl_values.append(np.sum(hist_fraud * np.log((hist_fraud + eps) / (hist_normal + eps))))
        kl = np.mean(kl_values)

        # AJUSTE LOG
        kl = np.log1p(abs(kl))
        wasser = np.log1p(abs(wasser))

        # INFORMAÇÃO MÚTUA
        mi = np.mean([mutual_info_score(pd.qcut(df_visu[f], q=10, duplicates='drop'), y_full) for f in [f1, f2, f3]])

        # ASSIMETRIA E CURTOSE
        skewness_score = 1 / (1 + np.mean([abs(df_visu[f].skew()) for f in [f1, f2, f3]]))
        kurtosis_score = 1 / (1 + np.mean([abs(df_visu[f].kurtosis()) for f in [f1, f2, f3]]))

        # OUTLIERS
        def contar_outliers(col):
            q1, q3 = col.quantile(0.25), col.quantile(0.75)
            iqr = q3 - q1
            return ((col < (q1 - 1.5 * iqr)) | (col > (q3 + 1.5 * iqr))).sum()

        outliers = np.mean([contar_outliers(df_visu[f]) for f in [f1, f2, f3]])
        outlier_score = 1 / (1 + outliers)

        # CORRELAÇÃO MÉDIA
        c12, _ = spearmanr(df_visu[f1], df_visu[f2])
        c13, _ = spearmanr(df_visu[f1], df_visu[f3])
        c23, _ = spearmanr(df_visu[f2], df_visu[f3])
        corr_score = 1 - np.mean([abs(c12), abs(c13), abs(c23)])

        return {
            'Features': f'{f1} | {f2} | {f3}',
            'Curva ROC': roc, 'KS': ks, 'divergencia_kl': kl, 'Wasser': wasser,
            'inf_mutua': mi, 'Overlap_GMM': overlap_score, 'Geo_Score': geo_score,
            'Skewness_Score': skewness_score, 'Kurtosis_Score': kurtosis_score,
            'Corr_Score': corr_score, 'Outlier_Score': outlier_score,
            'Silhouette': silhouette, 'Davies_Bouldin': db, 'Dist_Centroides': dist_centroides
        }
    except Exception as e:
        print(f'\nErro em {f1} | {f2} | {f3}: {e}')
        return None

# =========================================================
# BLOCO PRINCIPAL DE EXECUÇÃO
# =========================================================
if __name__ == '__main__':
    inicio = time.time()

    # =========================================================
    # DATASET PRINCIPAL
    # =========================================================
    print("\nCarregando dataset de crédito...")
    df = pd.read_csv('creditcard.csv')

    fraude = df[df['status_fraude'] == 1]
    normal = df[df['status_fraude'] == 0]

    df_visu = df.copy()

    df_cluster = df_visu.sample(frac=0.05, random_state=42)
    print(f'Amostra cluster: {len(df_cluster)} linhas')

    # =========================================================
    # EXTRAÇÃO DE TODAS AS FEATURES "V"
    # =========================================================
    features_all = [col for col in df_visu.columns if col.startswith('V')]
    print(f'\nTotal de features detectadas: {len(features_all)}')

    # =========================================================
    # COMBINAÇÕES 3x3 (Todas as possibilidades)
    # =========================================================
    triplas = list(combinations(features_all, 3))
    print(f'Total de combinações geradas: {len(triplas)} (deve ser 3276)')

    # =========================================================
    # PARALELIZAÇÃO COM JOBLIB (USANDO 7 NÚCLEOS)
    # =========================================================
    print(f'\nIniciando processamento paralelo com 7 núcleos...')
    
    resultados = Parallel(n_jobs=7, verbose=10)(
        delayed(processar_tripla)(f1, f2, f3, df_visu, df_cluster, fraude, normal) 
        for f1, f2, f3 in triplas
    )

    resultados = [r for r in resultados if r is not None]

    # =========================================================
    # DATAFRAME E SCALER DOS RESULTADOS FINAIS
    # =========================================================
    df_scores = pd.DataFrame(resultados)

    metricas = [
        'Curva ROC', 'KS', 'divergencia_kl', 'Wasser', 'inf_mutua', 
        'Overlap_GMM', 'Geo_Score', 'Skewness_Score', 'Kurtosis_Score', 
        'Corr_Score', 'Outlier_Score'
    ]

    df_scores[metricas] = df_scores[metricas].fillna(0)

    scaler = RobustScaler()
    df_norm = df_scores.copy()
    df_norm[metricas] = scaler.fit_transform(df_norm[metricas])

    for col in metricas:
        minimo = df_norm[col].min()
        maximo = df_norm[col].max()
        df_norm[col] = (df_norm[col] - minimo) / (maximo - minimo + 1e-9)

    # =========================================================
    # SCORE FINAL E RANKING DAS TRIPLAS
    # =========================================================
    df_norm['Score_Final'] = (
        df_norm['Curva ROC'] * 0.125 +
        df_norm['KS'] * 0.125 +
        df_norm['divergencia_kl'] * 0.105 +
        df_norm['Wasser'] * 0.105 +
        df_norm['inf_mutua'] * 0.085 +
        df_norm['Overlap_GMM'] * 0.085 +
        df_norm['Geo_Score'] * 0.13 +
        df_norm['Skewness_Score'] * 0.045 +
        df_norm['Kurtosis_Score'] * 0.045 +
        df_norm['Corr_Score'] * 0.025 +
        df_norm['Outlier_Score'] * 0.025
    )

    df_norm['Score_Final'] = df_norm['Score_Final'].fillna(0)

    ranking_triplas = df_norm.sort_values('Score_Final', ascending=False).reset_index(drop=True)
    ranking_triplas.insert(0, 'Posicao_Rank', ranking_triplas.index + 1)

    # =========================================================
    # EXIBIR E SALVAR
    # =========================================================
    pd.set_option('display.max_columns', None)
    display(ranking_triplas.head(30))

    # Salvando com um nome diferente para não apagar o das Top 10
    ranking_triplas.to_csv('3x3visu_scores_ALL.csv', index=False)

    fim = time.time()
    print(f'\nTempo total: {(fim - inicio)/60:.2f} minutos')
    print('\nCSV "3x3visu_scores_.csv" salvo com sucesso!')


Carregando dataset de crédito...
Amostra cluster: 14240 linhas

Total de features detectadas: 28
Total de combinações geradas: 3276 (deve ser 3276)

Iniciando processamento paralelo com 7 núcleos...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   4 tasks      | elapsed:   26.0s
[Parallel(n_jobs=7)]: Done  11 tasks      | elapsed:   26.4s
[Parallel(n_jobs=7)]: Done  18 tasks      | elapsed:   26.9s
[Parallel(n_jobs=7)]: Done  27 tasks      | elapsed:   27.7s
[Parallel(n_jobs=7)]: Done  36 tasks      | elapsed:   28.4s
[Parallel(n_jobs=7)]: Done  47 tasks      | elapsed:   29.2s
[Parallel(n_jobs=7)]: Done  58 tasks      | elapsed:   29.9s
[Parallel(n_jobs=7)]: Done  71 tasks      | elapsed:   30.6s
[Parallel(n_jobs=7)]: Done  84 tasks      | elapsed:   31.4s
[Parallel(n_jobs=7)]: Done  99 tasks      | elapsed:   32.2s
[Parallel(n_jobs=7)]: Done 114 tasks      | elapsed:   33.2s
[Parallel(n_jobs=7)]: Done 131 tasks      | elapsed:   33.9s
[Parallel(n_jobs=7)]: Done 148 tasks      | elapsed:   35.1s
[Parallel(n_jobs=7)]: Done 167 tasks      | elapsed:   36.2s
[Parallel(n_jobs=7)]: Done 186 tasks      | elapsed:   37.1s
[Parallel(

,Posicao_Rank,Features,Curva ROC,KS,divergencia_kl,Wasser,inf_mutua,Overlap_GMM,Geo_Score,Skewness_Score,Kurtosis_Score,Corr_Score,Outlier_Score,Silhouette,Davies_Bouldin,Dist_Centroides,Score_Final
0,1,V10 | V11 | V14,0.976472,0.987468,0.909498,0.892556,0.960329,0.909183,0.063611,0.453307,0.065600,0.845921,0.161385,0.544382,3.110538,1.156188,0.650420
1,2,V10 | V14 | V17,1.000000,0.982564,0.827291,0.970683,0.947149,0.930626,0.077397,0.248839,0.024430,0.739447,0.115327,0.602602,2.785491,1.454221,0.639947
2,3,V10 | V14 | V16,0.979502,0.956157,0.900769,0.902048,0.925292,0.909781,0.063662,0.393232,0.056990,0.848836,0.111261,0.574097,3.031767,1.123122,0.639773
3,4,V10 | V12 | V14,0.984062,1.000000,0.881138,0.952702,0.980036,0.792169,0.061557,0.319801,0.049628,0.819746,0.080879,0.445021,2.603261,1.165124,0.638341
4,5,V3 | V10 | V11,0.918041,0.924831,0.821573,0.894151,0.875366,0.960229,0.136390,0.432083,0.062658,0.870259,0.331770,0.614025,1.688181,2.948871,0.636580
5,6,V11 | V12 | V14,0.971860,0.978294,0.959610,0.907464,0.964314,0.782124,0.063430,0.369249,0.081960,0.670045,0.119737,0.428099,2.242806,1.202423,0.636554
6,7,V4 | V14 | V17,0.982390,0.965511,0.827065,0.944782,0.967113,0.909702,0.093003,0.269413,0.030743,0.742470,0.106772,0.518478,2.158200,1.912819,0.635889
7,8,V10 | V11 | V16,0.939838,0.917100,0.954302,0.813943,0.858469,0.925113,0.080452,0.546414,0.085030,0.872455,0.231009,0.585621,2.244675,1.510751,0.635848
8,9,V3 | V10 | V14,0.957705,0.963888,0.748724,0.970242,0.942189,0.931048,0.114707,0.321864,0.045753,0.832632,0.140777,0.570671,1.942062,2.431145,0.635706
9,10,V12 | V14 | V17,0.995388,0.973390,0.886573,0.983481,0.951134,0.848445,0.077468,0.211223,0.026643,0.651469,0.088470,0.481307,1.994749,1.507712,0.634690



Tempo total: 231.62 minutos

CSV "3x3visu_scores_.csv" salvo com sucesso!
